# YOLOの進化を動かして理解する

## 全体概要

YOLOは **画像全体を一度に見て物体の位置と種類を予測する one-stage detector** として始まりました。ただし、v1から現在までを管理する単一組織はありません。v5以降の番号は異なるチームが提案した系列を含むため、番号が大きいだけで常に同じ設計の直接後継、または全用途で優秀、という意味ではありません。

本教材は番号付きの主要リリースを v1→v13 の順に比較し、最後に Ultralytics の現行製品系列 YOLO26 を扱います。**v14〜v25という公開版があるわけではなく、YOLO26は年に合わせた命名**です。YOLOX、YOLO-NAS、RT-DETRなどの重要な派生は、番号順という今回の範囲から除外します。

### 学び方

1. 各節の「追加技術」と「使えるようになった場面」を読む
2. NumPyだけの小規模デモで中心概念を確認する
3. 対応する公式実装・重みで実画像推論を行う

小規模デモは本物のネットワーク精度を再現するものではなく、差分となる演算・データ表現を分離して観察する教材です。精度や速度の数値比較には、同じデータ、解像度、ハードウェア、測定条件が必要です。

In [ ]:
from pathlib import Path
import sys, json, subprocess

ROOT = Path.cwd()
if not (ROOT / 'common').exists():
    raise RuntimeError('このノートブックをリポジトリ直下から実行してください')
sys.path.insert(0, str(ROOT))

from common.versions import VERSIONS
from common.demos import run_demo
print('収録:', ', '.join(VERSIONS[v]['title'] for v in VERSIONS))

## 推論環境

技術デモのみ: `python -m pip install -r requirements-demo.txt`

画像推論も行う: `python -m pip install -r requirements.txt`

- v1〜v4: Darknet形式の `.cfg` と `.weights` を用意し、OpenCV DNNで実行
- v5/v8〜v12/YOLO26: `ultralytics` が読める対応 `.pt` を指定
- v6/v7/v13: 公式リポジトリごとに環境が異なるので、その推論コマンドを `--command` 経由で実行

推論だけならデータセットは不要で、手元のJPEG/PNGを1枚使えます。学習用データは末尾を参照してください。

## YOLOv1 (2016)

**新しく加わった技術:** 画像全体を1回のCNNで処理し、グリッドごとにbboxとクラスを同時回帰。

**その結果使いやすくなった場面:** 高速性が重要な動画・ロボットで、単段検出をリアルタイム利用。

**注意:** この節のデモは `grid` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/pjreddie/darknet) / [論文](https://arxiv.org/abs/1506.02640)

In [ ]:
# YOLOv1 の技術差分デモ
print(json.dumps(run_demo(1), ensure_ascii=False, indent=2))

### 実際の推論体験

まず別ターミナルで次を実行します（重み名・公式CLIの引数は、リンク先で取得した版に合わせてください）。

```bash
python yolov1.py --mode infer --image data/sample.jpg --config path/to/yolov1.cfg --weights path/to/yolov1.weights
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを端末に表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv2 / YOLO9000 (2017)

**新しく加わった技術:** anchor box、k-meansによるbox事前分布、BN、passthrough、階層分類。

**その結果使いやすくなった場面:** 形状の異なる物体と多数カテゴリ（検出・分類データの共同学習）。

**注意:** この節のデモは `anchors` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/pjreddie/darknet) / [論文](https://arxiv.org/abs/1612.08242)

In [ ]:
# YOLOv2 / YOLO9000 の技術差分デモ
print(json.dumps(run_demo(2), ensure_ascii=False, indent=2))

### 実際の推論体験

まず別ターミナルで次を実行します（重み名・公式CLIの引数は、リンク先で取得した版に合わせてください）。

```bash
python yolov2.py --mode infer --image data/sample.jpg --config path/to/yolov2.cfg --weights path/to/yolov2.weights
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを端末に表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv3 (2018)

**新しく加わった技術:** Darknet-53、3スケール予測、FPN的な特徴融合、独立ロジスティック分類。

**その結果使いやすくなった場面:** 小物体を含む監視映像や交通映像など、サイズ差の大きい対象。

**注意:** この節のデモは `multiscale` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/pjreddie/darknet) / [論文](https://arxiv.org/abs/1804.02767)

In [ ]:
# YOLOv3 の技術差分デモ
print(json.dumps(run_demo(3), ensure_ascii=False, indent=2))

### 実際の推論体験

まず別ターミナルで次を実行します（重み名・公式CLIの引数は、リンク先で取得した版に合わせてください）。

```bash
python yolov3.py --mode infer --image data/sample.jpg --config path/to/yolov3.cfg --weights path/to/yolov3.weights
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを端末に表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv4 (2020)

**新しく加わった技術:** CSPDarknet53、SPP+PAN、Mish、CIoU、Mosaic等のBoF/BoSを統合。

**その結果使いやすくなった場面:** 単一GPUで学習しつつ高精度な産業・交通向けリアルタイム検出。

**注意:** この節のデモは `mosaic` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/AlexeyAB/darknet) / [論文](https://arxiv.org/abs/2004.10934)

In [ ]:
# YOLOv4 の技術差分デモ
print(json.dumps(run_demo(4), ensure_ascii=False, indent=2))

### 実際の推論体験

まず別ターミナルで次を実行します（重み名・公式CLIの引数は、リンク先で取得した版に合わせてください）。

```bash
python yolov4.py --mode infer --image data/sample.jpg --config path/to/yolov4.cfg --weights path/to/yolov4.weights
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを端末に表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv5 (2020)

**新しく加わった技術:** PyTorch実装、CSP系backbone、PAN-FPN、AutoAnchor、使いやすい学習・配備API。

**その結果使いやすくなった場面:** 独自データの迅速な学習、ONNX/TensorRT等への実務配備。

**注意:** この節のデモは `focus` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/ultralytics/yolov5)

In [ ]:
# YOLOv5 の技術差分デモ
print(json.dumps(run_demo(5), ensure_ascii=False, indent=2))

### 実際の推論体験

まず別ターミナルで次を実行します（重み名・公式CLIの引数は、リンク先で取得した版に合わせてください）。

```bash
python yolov5.py --mode infer --image data/sample.jpg --weights yolov5nu.pt --save
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを端末に表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv6 (2022)

**新しく加わった技術:** RepVGG系の再パラメータ化、効率的decoupled head、SimOTA/TAL系割当。

**その結果使いやすくなった場面:** 物流・製造などGPU推論の低遅延が要求される産業用途。

**注意:** この節のデモは `reparam` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/meituan/YOLOv6) / [論文](https://arxiv.org/abs/2209.02976)

In [ ]:
# YOLOv6 の技術差分デモ
print(json.dumps(run_demo(6), ensure_ascii=False, indent=2))

### 実際の推論体験

まず別ターミナルで次を実行します（重み名・公式CLIの引数は、リンク先で取得した版に合わせてください）。

```bash
python yolov6.py --mode infer --image data/sample.jpg --weights path/to/model --command 'python path/to/official/infer.py --weights {weights} --source {image}'
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを端末に表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv7 (2022)

**新しく加わった技術:** E-ELAN、モデルscaling、予定再パラメータ化、trainable bag-of-freebies。

**その結果使いやすくなった場面:** エッジからGPUサーバまで、速度と精度の異なる制約へスケール。

**注意:** この節のデモは `elan` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/WongKinYiu/yolov7) / [論文](https://arxiv.org/abs/2207.02696)

In [ ]:
# YOLOv7 の技術差分デモ
print(json.dumps(run_demo(7), ensure_ascii=False, indent=2))

### 実際の推論体験

まず別ターミナルで次を実行します（重み名・公式CLIの引数は、リンク先で取得した版に合わせてください）。

```bash
python yolov7.py --mode infer --image data/sample.jpg --weights path/to/model --command 'python path/to/official/infer.py --weights {weights} --source {image}'
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを端末に表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv8 (2023)

**新しく加わった技術:** anchor-free decoupled head、C2f、タスク統合API（検出・分割・姿勢等）。

**その結果使いやすくなった場面:** 同じ操作体系で検出、segmentation、pose、trackingを試す製品開発。

**注意:** この節のデモは `anchor_free` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/ultralytics/ultralytics)

In [ ]:
# YOLOv8 の技術差分デモ
print(json.dumps(run_demo(8), ensure_ascii=False, indent=2))

### 実際の推論体験

まず別ターミナルで次を実行します（重み名・公式CLIの引数は、リンク先で取得した版に合わせてください）。

```bash
python yolov8.py --mode infer --image data/sample.jpg --weights yolov8n.pt --save
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを端末に表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv9 (2024)

**新しく加わった技術:** Programmable Gradient Information (PGI) と GELAN。

**その結果使いやすくなった場面:** 軽量モデルでも学習時の情報損失を抑えたいエッジ検出。

**注意:** この節のデモは `pgi` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/WongKinYiu/yolov9) / [論文](https://arxiv.org/abs/2402.13616)

In [ ]:
# YOLOv9 の技術差分デモ
print(json.dumps(run_demo(9), ensure_ascii=False, indent=2))

### 実際の推論体験

まず別ターミナルで次を実行します（重み名・公式CLIの引数は、リンク先で取得した版に合わせてください）。

```bash
python yolov9.py --mode infer --image data/sample.jpg --weights yolov9c.pt --save
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを端末に表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv10 (2024)

**新しく加わった技術:** consistent dual assignmentsによるend-to-end NMS-free検出と効率設計。

**その結果使いやすくなった場面:** NMS遅延を避けたいストリーミング・組込みリアルタイム推論。

**注意:** この節のデモは `nms_free` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/THU-MIG/yolov10) / [論文](https://arxiv.org/abs/2405.14458)

In [ ]:
# YOLOv10 の技術差分デモ
print(json.dumps(run_demo(10), ensure_ascii=False, indent=2))

### 実際の推論体験

まず別ターミナルで次を実行します（重み名・公式CLIの引数は、リンク先で取得した版に合わせてください）。

```bash
python yolov10.py --mode infer --image data/sample.jpg --weights yolov10n.pt --save
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを端末に表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLO11 (2024)

**新しく加わった技術:** C3k2/C2PSA等で特徴抽出とattentionを効率化しマルチタスク性能を改善。

**その結果使いやすくなった場面:** 限られた計算資源で検出・分割・姿勢・OBB・分類を横断利用。

**注意:** この節のデモは `attention` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/ultralytics/ultralytics)

In [ ]:
# YOLO11 の技術差分デモ
print(json.dumps(run_demo(11), ensure_ascii=False, indent=2))

### 実際の推論体験

まず別ターミナルで次を実行します（重み名・公式CLIの引数は、リンク先で取得した版に合わせてください）。

```bash
python yolov11.py --mode infer --image data/sample.jpg --weights yolo11n.pt --save
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを端末に表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv12 (2025)

**新しく加わった技術:** Area Attention (A2) と R-ELANでattentionをリアルタイム検出向けに最適化。

**その結果使いやすくなった場面:** 混雑場面など広い文脈が必要で、CNN並みの低遅延も必要な検出。

**注意:** この節のデモは `area_attention` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/sunsmarterjie/yolov12) / [論文](https://arxiv.org/abs/2502.12524)

In [ ]:
# YOLOv12 の技術差分デモ
print(json.dumps(run_demo(12), ensure_ascii=False, indent=2))

### 実際の推論体験

まず別ターミナルで次を実行します（重み名・公式CLIの引数は、リンク先で取得した版に合わせてください）。

```bash
python yolov12.py --mode infer --image data/sample.jpg --weights yolo12n.pt --save
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを端末に表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv13 (2025)

**新しく加わった技術:** HyperACEによるhypergraph特徴相関、FullPAD、DS-based label assignment。

**その結果使いやすくなった場面:** 遮蔽・密集・離れた部位の関係など高次の特徴相関が重要な場面。

**注意:** この節のデモは `hypergraph` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/iMoonLab/yolov13) / [論文](https://arxiv.org/abs/2506.17733)

In [ ]:
# YOLOv13 の技術差分デモ
print(json.dumps(run_demo(13), ensure_ascii=False, indent=2))

### 実際の推論体験

まず別ターミナルで次を実行します（重み名・公式CLIの引数は、リンク先で取得した版に合わせてください）。

```bash
python yolov13.py --mode infer --image data/sample.jpg --weights path/to/model --command 'python path/to/official/infer.py --weights {weights} --source {image}'
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを端末に表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## 補足: YOLO26 (Ultralytics product line) (2026)

**新しく加わった技術:** end-to-end NMS-free推論、DFL除去、ProgLoss+STAL、MuSGDによる学習最適化。

**その結果使いやすくなった場面:** CPU・エッジ機器への配備と、後処理を単純化した低遅延end-to-end推論。

**注意:** この節のデモは `deployment` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/ultralytics/ultralytics) / [公式ドキュメント](https://docs.ultralytics.com/models/yolo26/)

In [ ]:
# YOLO26 (Ultralytics product line) の技術差分デモ
print(json.dumps(run_demo(26), ensure_ascii=False, indent=2))

### 実際の推論体験

まず別ターミナルで次を実行します（重み名・公式CLIの引数は、リンク先で取得した版に合わせてください）。

```bash
python yolov26.py --mode infer --image data/sample.jpg --weights yolo26n.pt --save
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを端末に表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## 同じ画像で結果を比較する

以下では選んだ世代のスクリプトを実行します。`VERSION`, `IMAGE`, `WEIGHTS` を変更してください。v1〜v4では `CONFIG` も必要です。実行前に、上の各節にあるバックエンド条件を確認してください。

In [ ]:
VERSION = 8
IMAGE = 'data/sample.jpg'
WEIGHTS = 'yolov8n.pt'
CONFIG = None

cmd = [sys.executable, f'yolov{VERSION}.py', '--mode', 'infer', '--image', IMAGE, '--weights', WEIGHTS, '--save']
if CONFIG:
    cmd += ['--config', CONFIG]
print(' '.join(cmd))
# ダウンロードと推論を実行するときに次行のコメントを外す
# subprocess.run(cmd, check=True)

## データセット

**推論体験だけならダウンロード不要**です。`data/sample.jpg` に任意の写真を置いてください。

独自学習まで進む場合の小規模候補:

- [COCO128](https://docs.ultralytics.com/datasets/detect/coco128/): COCO train2017の先頭128枚。パイプライン確認向けで、性能評価には小さすぎます。
- [Pascal VOC](http://host.robots.ox.ac.uk/pascal/VOC/): v1/v2時代との比較に適した20クラス。
- [COCO](https://cocodataset.org/#download): 標準的な80クラス。容量・学習時間が大きい本評価向け。

データ利用条件と画像の権利を確認してください。世代比較では同じtrain/validation分割、入力解像度、augmentation、評価指標（通常 COCO mAP）を固定します。

## まとめ

- v1〜v3: 統一回帰 → anchor → multi-scale と、検出器の基本形を確立
- v4〜v7: 学習手法、特徴融合、再パラメータ化で速度精度比を改善
- v8〜v10: anchor-free、勾配設計、NMS-free end-to-endへ移行
- v11〜v13: attentionと高次特徴相関をリアルタイム制約へ導入
- YOLO26: 番号連番ではない製品系列として、配備とend-to-end効率を重視

「最新版」を選ぶ前に、ライセンス、対象タスク、対応export形式、実測レイテンシ、保守性を同じ重要度で確認してください。